In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

import joblib

RANDOM_STATE = 42

In [3]:
# Load full train with clusters
cluster0 = pd.read_csv("cluster_0.csv")

# Filter cluster 0
# cluster0 = df_all[df_all["cluster"] == 0].copy()
print(cluster0["Bankrupt?"].value_counts())   # should be 596 / 132

Bankrupt?
0    596
1    132
Name: count, dtype: int64


In [3]:
# Load the 40 features selected in 3.1
top40 = joblib.load("top_features_for_clustering.joblib")

X_sub = cluster0[top40].copy()
y_sub = cluster0["Bankrupt?"].copy()

print("X_sub shape:", X_sub.shape)
print("y_sub distribution:\n", y_sub.value_counts())

X_sub shape: (728, 40)
y_sub distribution:
 Bankrupt?
0    596
1    132
Name: count, dtype: int64


In [4]:
# Base models (non-parametric)
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

gb = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
)

et = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

base_estimators = [
    ("rf", rf),
    ("gb", gb),
    ("et", et),
]

# Meta-model
meta = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False,
)

# Full pipeline: scaling + stacking
stack_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("stack", stack),
    ]
)

In [5]:
recall1 = make_scorer(recall_score, pos_label=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(
    stack_pipeline,
    X_sub,
    y_sub,
    cv=cv,
    scoring=recall1,
    n_jobs=-1,
)

print("CV Eq(1) (recall y=1) mean:", cv_scores.mean())
print("CV scores:", cv_scores)

CV Eq(1) (recall y=1) mean: 0.7660968660968661
CV scores: [0.76923077 0.77777778 0.62962963 0.76923077 0.88461538]


In [6]:
# Fit pipeline on ALL cluster-0 rows
stack_pipeline.fit(X_sub, y_sub)

# Predictions on the original subgroup train data
y_pred = stack_pipeline.predict(X_sub)

cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
print("Confusion matrix [[FF, FT], [TF, TT]]:\n", cm)

FF, FT, TF, TT = cm.ravel()
print("FF:", FF, "FT:", FT, "TF:", TF, "TT:", TT)

eq1 = TT / (TF + TT) if (TF + TT) > 0 else 0.0
print("Eq(1) accuracy on cluster 0:", eq1)

# For Table 3:
N_features_cluster0 = len(top40)
print("Number of features used in subgroup model:", N_features_cluster0)

Confusion matrix [[FF, FT], [TF, TT]]:
 [[596   0]
 [  0 132]]
FF: 596 FT: 0 TF: 0 TT: 132
Eq(1) accuracy on cluster 0: 1.0
Number of features used in subgroup model: 40


In [7]:
cluster0_model = {
    "cluster_id": 0,
    "feature_names": top40,
    "pipeline": stack_pipeline,   # includes scaler + stacking model
}

joblib.dump(cluster0_model, "cluster0_stacking_PNH.joblib")
print("Saved cluster0_stacking_PNH.joblib")

Saved cluster0_stacking_PNH.joblib
